In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# normalization
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# golden rule
train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 249kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.63MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.8MB/s]


In [2]:
class FashionModelWithDropout(nn.Module):
    def __init__(self, dropout_p=0.5):
        super(FashionModelWithDropout, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 512)
        self.dropout1 = nn.Dropout(dropout_p)

        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(dropout_p)

        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

model = FashionModelWithDropout(dropout_p=0.5)

In [3]:
model = FashionModelWithDropout()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-3)

In [5]:
train_losses = []
val_losses = []
patience = 5
counter = 0
best_val_loss = float('inf')
epochs = 50

print("Starting training with Early Stopping...")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # --- VALIDATION PHASE ---
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for v_images, v_labels in val_loader:
            v_outputs = model(v_images)
            v_loss += criterion(v_outputs, v_labels).item()

    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = v_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"--> Best model saved at epoch {epoch+1}")
    else:
        counter += 1
        print(f"No improvement for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered! Training finished at epoch {epoch+1}.")
            break

Starting training with Early Stopping...
Epoch 1 - Train Loss: 0.7133 - Val Loss: 0.5658
--> Best model saved at epoch 1
Epoch 2 - Train Loss: 0.6177 - Val Loss: 0.5105
--> Best model saved at epoch 2
Epoch 3 - Train Loss: 0.5659 - Val Loss: 0.4791
--> Best model saved at epoch 3
Epoch 4 - Train Loss: 0.5309 - Val Loss: 0.4561
--> Best model saved at epoch 4
Epoch 5 - Train Loss: 0.5056 - Val Loss: 0.4419
--> Best model saved at epoch 5
Epoch 6 - Train Loss: 0.4855 - Val Loss: 0.4275
--> Best model saved at epoch 6
Epoch 7 - Train Loss: 0.4703 - Val Loss: 0.4146
--> Best model saved at epoch 7
Epoch 8 - Train Loss: 0.4560 - Val Loss: 0.4094
--> Best model saved at epoch 8
Epoch 9 - Train Loss: 0.4439 - Val Loss: 0.3958
--> Best model saved at epoch 9
Epoch 10 - Train Loss: 0.4355 - Val Loss: 0.3926
--> Best model saved at epoch 10
Epoch 11 - Train Loss: 0.4247 - Val Loss: 0.3852
--> Best model saved at epoch 11
Epoch 12 - Train Loss: 0.4182 - Val Loss: 0.3856
No improvement for 1 epoch